In [1]:
#Two Agents with OpenAI API Key.

In [2]:
#!pip install autogen

In [3]:
# 1: CHECK AUTOGEN INSTALLATION AND VERSION

import autogen
# import: loads a Python module so you can use it.
# autogen: the imported module name for the AutoGen package.
# If the package is missing, Python raises ModuleNotFoundError.

print("AutoGen version:", getattr(autogen, "__version__", "Version not exposed"))
# print(): displays the result in the notebook.
# "AutoGen version:": descriptive text shown before the version.
# getattr(): reads an attribute when it exists, or returns a fallback.
# autogen: the imported package.
# "__version__": the attribute commonly used to expose a package version.
# "Version not exposed": fallback text if that attribute is unavailable.

AutoGen version: 0.14.1


In [4]:
#!pip install agentops

In [5]:
#import agentops
#agentops.init("Put your key")

In [6]:
# 2: CREATE AN AUTOGEN CODE EXECUTOR AGENT

from pathlib import Path
# from: imports an object from a module.
# pathlib: Python's filesystem path-handling module.
# import: selects an object to bring into the current namespace.
# Path: class for creating and managing file and directory paths.

from autogen import ConversableAgent
# from: specifies the source module.
# autogen: the AutoGen package.
# import: requests a named object from that package.
# ConversableAgent: class for creating an agent that can participate
# in conversations with other agents.

from autogen.coding import CodeBlock, LocalCommandLineCodeExecutor
# autogen: the main package.
# .coding: selects the coding submodule.
# import: imports named objects from that submodule.
# CodeBlock: represents a block of code in AutoGen's coding utilities.
# LocalCommandLineCodeExecutor: class that executes code locally
# through a command-line execution process.

# STEP 1: CREATE A WORKING DIRECTORY
# STEP: marks a stage in the setup.
# 1: the first stage.
# CREATE A WORKING DIRECTORY: describes the purpose of this section.

work_dir = Path("coding")
# work_dir: variable name for the working-directory path.
# =: assignment operator; stores the right-hand value in the variable.
# Path: creates a Path object.
# "coding": string naming the folder relative to the current directory.
# Parentheses: pass the string as an argument to Path().

work_dir.mkdir(exist_ok=True)
# work_dir: refers to the Path object created above.
# .: accesses a method belonging to that object.
# mkdir: method that creates a directory.
# (): calls the method.
# exist_ok: parameter controlling what happens if the directory exists.
# =: assigns the argument value.
# True: allows an existing directory without raising an error.

# STEP 2: CONFIGURE THE CODE EXECUTOR
# CONFIGURE: means set up the executor's options.

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)
# executor: variable that stores the executor object.
# =: assigns the object to the variable.
# LocalCommandLineCodeExecutor: creates a local code-execution utility.
# (): calls the class constructor.
# work_dir=: names the constructor parameter.
# work_dir: supplies the directory path created in Step 1.

# STEP 3: CREATE AGENT 1
# Agent 1: labels the first agent in this example.

code_executor_agent = ConversableAgent(
    # code_executor_agent: variable name for the agent object.
    # =: assigns the constructed agent to the variable.
    # ConversableAgent: class used to construct the agent.
    # (: begins the argument list.

    name="code_executor_agent",
    # name: parameter that identifies the agent.
    # =: assigns a value to the parameter.
    # "code_executor_agent": the agent's name as a string.
    # ,: separates this argument from the next one.

    llm_config=False,
    # llm_config: parameter for language-model configuration.
    # =: assigns a value.
    # False: disables the agent's own LLM configuration.
    # This agent is intended to execute code rather than generate it.

    code_execution_config={
        # code_execution_config: parameter for code-execution settings.
        # =: assigns the dictionary that follows.
        # {: starts a Python dictionary.

        "executor": executor,
        # "executor": dictionary key identifying the executor setting.
        # :: separates the dictionary key from its value.
        # executor: the LocalCommandLineCodeExecutor object from Step 2.
        # ,: separates dictionary entries.

    },
    # }: ends the dictionary.
    # ,: separates this argument from the next one.

    human_input_mode="NEVER",
    # human_input_mode: parameter controlling interactive human input.
    # =: assigns a value to the parameter.
    # "NEVER": tells this agent not to request interactive human input.
    # ,: separates this argument from the closing parenthesis.

)
# ): closes the ConversableAgent constructor call.
# The resulting agent object is stored in code_executor_agent.

# STEP 4: DISPLAY A CONFIRMATION

print("AutoGen code executor agent created.")
# print: built-in Python function that displays output.
# (: begins the function's argument list.
# "AutoGen code executor agent created.": message to display.
# ): ends the function call.

AutoGen code executor agent created.


In [7]:
# 3: FIX API CREDENTIALS AND CREATE THE CODE WRITER AGENT

# STEP 1: IMPORT REQUIRED MODULES

import os
# import: loads a Python module.
# os: provides operating-system functions, including environment variables.

from dotenv import load_dotenv
# from dotenv: accesses the python-dotenv package.
# import load_dotenv: imports the function used to load a .env file.

from autogen import ConversableAgent
# autogen: the AutoGen package.
# ConversableAgent: the class used to create a conversational agent.

# STEP 2: LOAD THE .ENV FILE

load_dotenv(override=True)
# load_dotenv(): reads environment variables from a .env file.
# override=True: allows values in the .env file to replace existing values.
# The .env file must be in the current working directory or a searched parent directory.

# STEP 3: RETRIEVE AND VALIDATE THE API KEY

api_key = os.getenv("OPENAI_API_KEY")
# api_key: variable that stores the API key.
# os.getenv(): retrieves an environment variable.
# "OPENAI_API_KEY": the environment-variable name containing your API key.

if not api_key:
    # if: checks whether a condition is true.
    # not api_key: checks whether the key is missing or empty.

    raise ValueError(
        "API key missing. Check that your .env file contains "
        "OPENAI_API_KEY=your_actual_api_key"
    )
    # raise: stops execution and reports an error.
    # ValueError: indicates missing or invalid configuration.
    # The message tells you what to check without revealing your key.

print("API key found.")
# print(): displays a message in the notebook.
# This confirms that a value exists without printing the secret.

# STEP 4: DEFINE THE CODE WRITER'S SYSTEM MESSAGE

code_writer_system_message = """
You are a helpful code-writing agent.

Help solve tasks using Python code or shell scripts.
Provide complete, executable code when coding is required.
Explain the plan when necessary and solve tasks step by step.
Clearly distinguish code-based steps from explanations.
Do not execute code yourself.
Never reveal API keys, passwords, or other secret credentials.
Review generated code for potentially destructive actions.
"""
# code_writer_system_message: stores the agent's instructions.
# Triple quotes: allow the instructions to span multiple lines.
# The text defines the agent's role and expected behavior.

# STEP 5: SET THE REQUESTED MODEL

MODEL_NAME = "gpt-5.6-luna"
# MODEL_NAME: variable containing the requested model identifier.
# "gpt-5.6-luna": the model name you requested.
# The API provider must recognize this exact model ID.

# STEP 6: CREATE AGENT 2 — THE CODE WRITER

code_writer_agent = ConversableAgent(
    # code_writer_agent: variable that stores the created agent.
    # ConversableAgent(...): constructs a conversational agent.

    name="code_writer",
    # name: parameter specifying the agent's name.
    # "code_writer": identifies this agent in the workflow.

    system_message=code_writer_system_message,
    # system_message: supplies instructions to the agent.
    # code_writer_system_message: the instruction string defined above.

    llm_config={
        # llm_config: dictionary containing language-model settings.

        "config_list": [
            # config_list: list of model configurations.

            {
                "model": MODEL_NAME,
                # model: specifies the requested language model.
                # MODEL_NAME: uses the identifier defined in Step 5.

                "api_key": api_key,
                # api_key: supplies the authentication credential.
                # api_key: the value retrieved from the environment.
            }
        ],
    },

    code_execution_config=False,
    # False: prevents this agent from executing code itself.

    max_consecutive_auto_reply=2,
    # 2: allows up to two consecutive automatic replies.

    human_input_mode="NEVER",
    # NEVER: prevents requests for interactive human input.
)

# STEP 7: CONFIRM AGENT CREATION

print("Code writer agent created.")
# Displays a confirmation if the constructor completes successfully.

print("Requested model:", MODEL_NAME)
# Displays the requested model name without exposing the API key.

API key found.
Code writer agent created.
Requested model: gpt-5.6-luna


In [8]:
# 4: USE TWO AUTOGEN AGENTS TO CALCULATE PERMUTATIONS

# STEP 1: IMPORT REQUIRED MODULES

import pprint
# import: loads a Python module.
# pprint: provides readable formatting for Python objects.

from collections import Counter
# from: imports a specific object from a module.
# collections: Python's built-in collection utilities.
# Counter: counts the frequency of each letter.

from math import factorial
# math: Python's built-in mathematical module.
# factorial: calculates n! (n factorial).

# STEP 2: DEFINE THE WORD

word = "ALGEBRA"
# word: variable that stores the input word.
# =: assigns the value on the right to the variable on the left.
# "ALGEBRA": the word whose distinct permutations we want to calculate.

# STEP 3: ASK THE CODE-WRITER AGENT TO GENERATE CODE

chat_result = code_executor_agent.initiate_chat(
    # chat_result: stores the result returned by the agent conversation.
    # code_executor_agent: the agent that coordinates the conversation.
    # initiate_chat(): starts a conversation with another agent.

    recipient=code_writer_agent,
    # recipient: identifies the agent receiving the request.
    # code_writer_agent: the agent instructed to generate Python code.

    message=(
        f"Write executable Python code to calculate the number of "
        f"distinct permutations of the word {word}. "
        "Account for repeated letters. "
        "Return only a valid Python code block labelled python. "
        "The code must print the final integer result."
    ),
    # message: contains the instructions sent to the code-writer agent.
    # f"...": an f-string inserts the value of word into the message.
    # The instructions request Python code and account for repeated letters.
)
# ): closes the initiate_chat() function call.

# STEP 4: DISPLAY THE AGENT CONVERSATION RESULT

pprint.pprint(chat_result.chat_history)
# pprint.pprint(): displays the conversation history in readable form.
# chat_result.chat_history: contains the messages exchanged by the agents.
# Reviewing this history helps identify whether code execution succeeded.

# STEP 5: COUNT EACH LETTER IN THE WORD

letter_counts = Counter(word)
# Counter(word): counts the frequency of each character in word.
# For ALGEBRA, A appears twice; L, G, E, B and R appear once.
# letter_counts: stores these frequencies.

# STEP 6: CALCULATE THE DISTINCT PERMUTATIONS

distinct_permutations = factorial(len(word))
# len(word): returns the number of characters in the word, which is 7.
# factorial(7): calculates 7! = 5040.
# distinct_permutations: initially stores the total arrangements
# before removing duplicates caused by repeated letters.

for count in letter_counts.values():
    # for: repeats the indented statement for each letter frequency.
    # letter_counts.values(): retrieves the counts of all letters.
    # count: stores one letter's frequency during each iteration.

    distinct_permutations //= factorial(count)
    # factorial(count): calculates the factorial of the letter frequency.
    # //=: divides by that factorial and updates the variable.
    # This removes duplicate arrangements caused by repeated letters.

# STEP 7: DISPLAY THE VERIFIED ANSWER

print("Word:", word)
# print(): displays the input word.

print("Letter frequencies:", dict(letter_counts))
# dict(): converts the Counter object into a standard dictionary.
# The dictionary shows how often each letter appears.

print("Distinct permutations:", distinct_permutations)
# Displays the final calculated integer.

# STEP 8: VERIFY THE EXPECTED RESULT

assert distinct_permutations == 2520, "Permutation result is incorrect."
# assert: checks whether a condition is True.
# ==: compares the calculated result with 2520.
# If the condition is False, Python raises an AssertionError.

print("Verification successful.")
# Displays this confirmation if the assertion passes.

code_executor_agent (to code_writer):

Write executable Python code to calculate the number of distinct permutations of the word ALGEBRA. Account for repeated letters. Return only a valid Python code block labelled python. The code must print the final integer result.

--------------------------------------------------------------------------------
[autogen.oai.client: 10-28 18:22:59] {734} WARNING - Model gpt-5.6-luna is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
code_writer (to code_executor_agent):

```python
from math import factorial
from collections import Counter

word = "ALGEBRA"
counts = Counter(word)

result = factorial(len(word))
for count in counts.values():
    result //= factorial(count)

print(result)
```

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING CODE BLOCK (inferred language is python)...
code_executor_ag

In [9]:
# 5: INSPECT THE CHAT RESULT

dir(chat_result)
# dir(): built-in Python function that lists an object's available attributes and methods.
# chat_result: variable that stores the result returned by the AutoGen conversation.
# ( ): parentheses pass chat_result as the argument to dir().
# The output helps you discover which properties and methods are available.

['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'chat_history',
 'chat_id',
 'cost',
 'human_input',
 'summary']

In [10]:
# 6: DISPLAY AUTOGEN CHAT RESULT ATTRIBUTES

# STEP 1: CHECK THAT chat_result EXISTS

print("Chat result variable exists:", "chat_result" in globals())
# print(): displays a message and its value.
# "chat_result" in globals(): checks whether chat_result is defined
# in the current Python kernel.

# STEP 2: DISPLAY THE CHAT HISTORY

if "chat_result" in globals():
    print("\n--- CHAT HISTORY ---")
    print(chat_result.chat_history)
# if: executes the indented block only when the condition is True.
# \n: inserts a new line before the heading.
# chat_result.chat_history: retrieves the conversation messages.
# print(): displays the retrieved messages.

# STEP 3: DISPLAY THE CHAT ID

    print("\n--- CHAT ID ---")
    print(chat_result.chat_id)
# chat_id: identifies the conversation, if supported by this result object.

# STEP 4: DISPLAY COST INFORMATION

    print("\n--- COST ---")
    print(chat_result.cost)
# cost: displays cost-related information recorded by AutoGen.

# STEP 5: DISPLAY HUMAN INPUT INFORMATION

    print("\n--- HUMAN INPUT ---")
    print(chat_result.human_input)
# human_input: displays recorded human-input information, if available.

# STEP 6: DISPLAY THE SUMMARY

    print("\n--- SUMMARY ---")
    print(chat_result.summary)
# summary: displays the conversation summary, if available.

else:
    print("chat_result is not defined. Run the agent conversation cell first.")
# else: runs when chat_result does not exist.
# Run the cell that creates chat_result before running this cell.

Chat result variable exists: True

--- CHAT HISTORY ---
[{'content': 'Write executable Python code to calculate the number of distinct permutations of the word ALGEBRA. Account for repeated letters. Return only a valid Python code block labelled python. The code must print the final integer result.', 'role': 'assistant', 'name': 'code_executor_agent'}, {'content': '```python\nfrom math import factorial\nfrom collections import Counter\n\nword = "ALGEBRA"\ncounts = Counter(word)\n\nresult = factorial(len(word))\nfor count in counts.values():\n    result //= factorial(count)\n\nprint(result)\n```', 'role': 'user', 'name': 'code_writer'}, {'content': 'exitcode: 0 (execution succeeded)\nCode output: 2520\n', 'role': 'assistant', 'name': 'code_executor_agent'}, {'content': '```python\nfrom math import factorial\nfrom collections import Counter\n\nword = "ALGEBRA"\ncounts = Counter(word)\n\nresult = factorial(len(word))\nfor count in counts.values():\n    result //= factorial(count)\n\nprint